In [0]:
from pyspark.sql import functions as F

UK = "Europe/London"
KEYS = ["settlement_date", "settlement_period"]
LOC_WIND = ["wind_off_hornsea", "wind_off_dogger", "wind_off_eastanglia",
            "wind_off_seagreen", "wind_off_moray", "wind_on_scotland"]
LOC_SOLAR = ["solar_cambs", "solar_kent", "solar_somerset"]
LOC_TEMP = ["temp_london", "temp_birmingham", "temp_manchester"]

dem = spark.table("workspace.silver.demand_forecast_asof")
wind = spark.table("workspace.silver.wind_forecast_asof")
price = spark.table("workspace.silver.system_prices")
settled = spark.table("workspace.silver.settled_prices")
weather = spark.table("workspace.silver.weather_forecast_asof")
demand = spark.table("workspace.silver.demand_outturn")
fuel_wind = spark.table("workspace.silver.generation_by_fuel").filter("fuel_type = 'WIND'")

# 1. Backbone (every half hour of every target day), Elexon forecasts, target
g = (dem.select(*KEYS, "start_time_utc",
                F.col("national_demand_mw").alias("fc_national_demand_mw"),
                F.col("transmission_demand_mw").alias("fc_transmission_demand_mw"),
                F.col("age_at_decision_hours").alias("fc_demand_age_h"),
                "decision_time_utc")
     .withColumn("wind_hour_utc", F.date_trunc("hour", "start_time_utc"))
     .join(wind.select(F.col("start_time_utc").alias("wind_hour_utc"),
                       F.col("wind_generation_mw").alias("fc_wind_mw")), "wind_hour_utc", "left")
     .drop("wind_hour_utc")
     # Target = the settled price: what a battery is actually paid on
     .join(settled.select(*KEYS,
                          F.col("system_sell_price").alias("target_price"),
                          F.col("run_type").alias("target_run_type"),
                          F.col("is_settled").alias("target_is_settled"),
                          F.col("is_final").alias("target_is_final")), KEYS, "left")
     # The market feed price, kept to measure how much the switch matters. Never a feature.
     .join(price.select(*KEYS, F.col("system_sell_price").alias("feed_price")), KEYS, "left"))

# 2. Weather: one row per target day and hour, one column per place and variable
wide = (weather.groupBy("target_date", "valid_time_utc")
        .pivot("location", LOC_WIND + LOC_SOLAR + LOC_TEMP)
        .agg(F.first("temperature_2m").alias("temp"), F.first("shortwave_radiation").alias("rad"),
             F.first("wind_speed_100m").alias("w100"), F.first("cloud_cover").alias("cloud")))
snapshots = wide.select(
    F.col("target_date").alias("settlement_date"), F.col("valid_time_utc").alias("snap_hour"),
    *[F.col(f"{l}_w100").alias(f"w100_{l}") for l in LOC_WIND],
    *[F.col(f"{l}_cloud").alias(f"cloud_{l}") for l in LOC_SOLAR],
    *[F.col(f"{l}_temp").alias(f"temp_{l}") for l in LOC_TEMP])
sunshine = wide.select(
    F.col("target_date").alias("settlement_date"), F.col("valid_time_utc").alias("rad_hour"),
    *[F.col(f"{l}_rad").alias(f"rad_{l}") for l in LOC_SOLAR])
age = weather.groupBy(F.col("target_date").alias("settlement_date")).agg(
    F.max("run_age_at_decision_hours").alias("weather_run_age_h"))

def mean_of(cols):
    return sum(F.col(c) for c in cols) / len(cols)

g = (g
     .withColumn("snap_hour", F.date_trunc("hour", F.col("start_time_utc") + F.expr("INTERVAL 30 MINUTES")))
     .withColumn("rad_hour", F.date_trunc("hour", "start_time_utc") + F.expr("INTERVAL 1 HOUR"))
     .join(snapshots, ["settlement_date", "snap_hour"], "left")
     .join(sunshine, ["settlement_date", "rad_hour"], "left")
     .join(age, "settlement_date", "left")
     .drop("snap_hour", "rad_hour")
     .withColumn("w100_mean", mean_of([f"w100_{l}" for l in LOC_WIND]))
     .withColumn("rad_mean", mean_of([f"rad_{l}" for l in LOC_SOLAR]))
     .withColumn("cloud_mean", mean_of([f"cloud_{l}" for l in LOC_SOLAR]))
     .withColumn("temp_mean", mean_of([f"temp_{l}" for l in LOC_TEMP])))

# 3. Lags: built from the FEED price and actuals, only if published before the decision
targets = g.select(*KEYS, "decision_time_utc")

def lagged(src, value_col, days, name):
    """Value from the same settlement period `days` earlier, only if published before the decision."""
    s = src.select(F.col("settlement_date").alias("_src_date"),
                   F.col("settlement_period").alias("_src_period"),
                   F.col(value_col).alias("_value"),
                   F.col("published_at_utc").alias("_published"))
    j = targets.join(s, (F.date_sub(targets.settlement_date, days) == s._src_date)
                        & (targets.settlement_period == s._src_period), "left")
    return j.select(*KEYS,
                    F.when(F.col("_published") <= F.col("decision_time_utc"), F.col("_value")).alias(name),
                    (F.col("_value").isNotNull() & (F.col("_published") > F.col("decision_time_utc")))
                    .cast("int").alias(f"_{name}_blanked"))

lags = [
    lagged(price, "system_sell_price", 3, "price_lag3"),
    lagged(price, "system_sell_price", 7, "price_lag7"),
    lagged(demand, "initial_demand_outturn", 2, "demand_lag2"),
    lagged(fuel_wind, "generation", 2, "wind_actual_lag2"),
]

# Daily summary of feed prices 3 days earlier, only those published before the decision
day3 = (targets.select("settlement_date", "decision_time_utc").distinct()
        .join(price.select(F.col("settlement_date").alias("_d"), "system_sell_price", "published_at_utc"),
              F.date_sub(F.col("settlement_date"), 3) == F.col("_d"), "left")
        .filter(F.col("published_at_utc") <= F.col("decision_time_utc"))
        .groupBy("settlement_date")
        .agg(F.avg("system_sell_price").alias("price_d3_mean"),
             F.min("system_sell_price").alias("price_d3_min"),
             F.max("system_sell_price").alias("price_d3_max"),
             F.count("*").alias("price_d3_count")))

for lag in lags:
    g = g.join(lag, KEYS, "left")

too_few = F.col("price_d3_count") < 43   # fewer than 90% of a 48 half hour day
local = F.from_utc_timestamp("start_time_utc", UK)
g = (g.join(day3, "settlement_date", "left")
      .withColumn("price_d3_mean", F.when(too_few, None).otherwise(F.col("price_d3_mean")))
      .withColumn("price_d3_min", F.when(too_few, None).otherwise(F.col("price_d3_min")))
      .withColumn("price_d3_max", F.when(too_few, None).otherwise(F.col("price_d3_max")))
      # 4. Calendar
      .withColumn("uk_hour", F.hour(local) + F.minute(local) / 60)
      .withColumn("day_of_week", F.dayofweek(local))            # 1 = Sunday, 7 = Saturday
      .withColumn("is_weekend", F.dayofweek(local).isin(1, 7).cast("int"))
      .withColumn("month", F.month(local)))

blanked = g.select(*[F.sum(c).alias(c) for c in g.columns if c.endswith("_blanked")]).collect()[0].asDict()
g = g.drop(*[c for c in g.columns if c.endswith("_blanked")])

# 5. Save once
(g.write.format("delta").mode("overwrite").option("overwriteSchema", True)
  .saveAsTable("workspace.gold.price_features"))

t = spark.table("workspace.gold.price_features")
print("rows:", t.count(), "| columns:", len(t.columns))
print("blanked because published after the decision:", blanked)

rows: 35712 | columns: 45
blanked because published after the decision: {'_price_lag3_blanked': 1074, '_price_lag7_blanked': 1056, '_demand_lag2_blanked': 0, '_wind_actual_lag2_blanked': 0}


In [0]:
from pyspark.sql import functions as F

KEYS = ["settlement_date", "settlement_period"]
t = spark.table("workspace.gold.price_features")
print(len(t.columns), "columns:", t.columns)

# 1. Shape and target
display(t.select(
    F.count("*").alias("rows"),
    F.countDistinct(*KEYS).alias("unique_half_hours"),
    F.sum(F.col("target_price").isNull().cast("int")).alias("missing_target"),
    F.sum(F.col("feed_price").isNull().cast("int")).alias("missing_feed"),
    F.sum((F.abs(F.col("target_price") - F.col("feed_price")) > 0.005).cast("int")).alias("target_differs_from_feed"),
    F.sum(F.col("target_is_final").cast("int")).alias("final"),
    F.sum((F.col("target_is_settled") & ~F.col("target_is_final")).cast("int")).alias("settled_not_final"),
    F.sum((~F.col("target_is_settled")).cast("int")).alias("not_settled_yet"),
))

# 2. The 10am rule
display(t.select(
    F.sum((F.col("decision_time_utc") >= F.col("start_time_utc")).cast("int")).alias("decision_not_before_half_hour"),
    F.sum((F.col("fc_demand_age_h") < 0).cast("int")).alias("demand_fc_from_future"),
    F.sum((F.col("weather_run_age_h") < 6).cast("int")).alias("weather_run_too_fresh"),
    F.min("fc_demand_age_h").alias("min_demand_fc_age_h"),
    F.min("weather_run_age_h").alias("min_weather_age_h"),
))

# 3. Correlations with the new target
checks = ["fc_national_demand_mw", "fc_wind_mw", "w100_mean", "rad_mean", "temp_mean",
          "price_lag3", "price_lag7", "price_d3_mean", "demand_lag2", "wind_actual_lag2"]
display(t.filter("target_price IS NOT NULL").select(
    *[F.round(F.corr("target_price", c), 3).alias(c) for c in checks]))

# 4. Did anything change apart from the target? Compare with Gold from before tonight
try:
    old = spark.sql("SELECT * FROM workspace.gold.price_features TIMESTAMP AS OF '2026-10-23 22:00:00'")
    numeric = {f.name for f in t.schema.fields
               if f.dataType.typeName() in ("double", "float", "integer", "long", "short", "decimal")}
    shared = [c for c in old.columns if c in t.columns and c not in KEYS + ["target_price"]]

    def differs(c, new_col, old_col):
        n, o = F.col(f"n.{new_col}"), F.col(f"o.{old_col}")
        if c in numeric:
            return (n.isNull() != o.isNull()) | (F.abs(n - o) > 1e-9)
        return ~n.eqNullSafe(o)

    j = t.alias("n").join(old.alias("o"), KEYS, "full")
    display(j.select(
        F.count("*").alias("rows_joined"),
        *[F.sum(F.coalesce(differs(c, c, c), F.lit(False)).cast("int")).alias(c) for c in shared],
        F.sum(F.coalesce(differs("feed_price", "feed_price", "target_price"), F.lit(False)).cast("int"))
         .alias("feed_vs_old_target"),
    ))
except Exception as e:
    print("Old version not available:", str(e)[:300])

45 columns: ['settlement_date', 'settlement_period', 'start_time_utc', 'fc_national_demand_mw', 'fc_transmission_demand_mw', 'fc_demand_age_h', 'decision_time_utc', 'fc_wind_mw', 'target_price', 'target_run_type', 'target_is_settled', 'target_is_final', 'feed_price', 'w100_wind_off_hornsea', 'w100_wind_off_dogger', 'w100_wind_off_eastanglia', 'w100_wind_off_seagreen', 'w100_wind_off_moray', 'w100_wind_on_scotland', 'cloud_solar_cambs', 'cloud_solar_kent', 'cloud_solar_somerset', 'temp_temp_london', 'temp_temp_birmingham', 'temp_temp_manchester', 'rad_solar_cambs', 'rad_solar_kent', 'rad_solar_somerset', 'weather_run_age_h', 'w100_mean', 'rad_mean', 'cloud_mean', 'temp_mean', 'price_lag3', 'price_lag7', 'demand_lag2', 'wind_actual_lag2', 'price_d3_mean', 'price_d3_min', 'price_d3_max', 'price_d3_count', 'uk_hour', 'day_of_week', 'is_weekend', 'month']


rows,unique_half_hours,missing_target,missing_feed,target_differs_from_feed,final,settled_not_final,not_settled_yet
35712,35712,0,49,1521,16272,19440,0


decision_not_before_half_hour,demand_fc_from_future,weather_run_too_fresh,min_demand_fc_age_h,min_weather_age_h
0,0,0,0.05,9.0


fc_national_demand_mw,fc_wind_mw,w100_mean,rad_mean,temp_mean,price_lag3,price_lag7,price_d3_mean,demand_lag2,wind_actual_lag2
0.273,-0.239,-0.232,-0.137,-0.011,0.167,0.164,0.148,0.167,-0.028


rows_joined,start_time_utc,fc_national_demand_mw,fc_transmission_demand_mw,fc_demand_age_h,decision_time_utc,fc_wind_mw,w100_wind_off_hornsea,w100_wind_off_dogger,w100_wind_off_eastanglia,w100_wind_off_seagreen,w100_wind_off_moray,w100_wind_on_scotland,cloud_solar_cambs,cloud_solar_kent,cloud_solar_somerset,temp_temp_london,temp_temp_birmingham,temp_temp_manchester,rad_solar_cambs,rad_solar_kent,rad_solar_somerset,weather_run_age_h,w100_mean,rad_mean,cloud_mean,temp_mean,price_lag3,price_lag7,demand_lag2,wind_actual_lag2,price_d3_mean,price_d3_min,price_d3_max,price_d3_count,uk_hour,day_of_week,is_weekend,month,feed_vs_old_target
35712,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
